# DC-Gaussian — Reflection Removal Notebook

**NeurIPS 2024 · Wang et al., "DC-Gaussian: Improving 3D Gaussian Splatting for Reflective Dash Cam Videos"**

---

### What this notebook does

```
SEQ_ROOT/
  seq_001/  *.jpg / *.png (sorted by name)
  seq_002/  ...
         ⬇  COLMAP → DC-Gaussian train → DC-Gaussian render
OUTPUT_ROOT/
  seq_001/  <original_filenames>.png   ← reflection-free
  seq_002/  ...
```

### Assumptions
- You are running this inside the **`dcgaussian`** conda environment (created by `setup_dc_gaussian.sh`).
- `COLMAP` is on your `PATH` (or set `COLMAP_BIN` to its full path).
- DC-Gaussian is cloned at `DC_GAUSSIAN_DIR`.
- GPU with ≥ 8 GB VRAM is available.

### Run order
`Cell 1 → Cell 2 → Cell 3 (preflight) → Cell 4 (pipeline) → Cell 5 (summary)`

## Cell 1 — Imports

In [ ]:
import os
import sys
import glob
import shutil
import subprocess
import logging
from pathlib import Path
from datetime import datetime

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout)],
    force=True,
)
log = logging.getLogger("dc_gaussian")
print("Imports OK ✓")
print(f"Python : {sys.executable}")


## Cell 2 — Configuration

Edit the paths below before running.  Everything else auto-derives from them.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  PATHS  — edit these
# ══════════════════════════════════════════════════════════════════════════════

# Input: root folder that contains one sub-folder per sequence
SEQ_ROOT        = "/STORAGE/Arka/a_MDE_Code/3_Datasets/Sample_Dataset/Seq_images"

# Output: reflection-free PNGs land here, preserving subfolder & file names
OUTPUT_ROOT     = "/STORAGE/Arka/a_MDE_Code/2_Corrections/Output"

# DC-Gaussian repo (cloned by setup_dc_gaussian.sh)
DC_GAUSSIAN_DIR = "/STORAGE/Arka/DC-Gaussian"

# Intermediate files (COLMAP sparse, undistorted images, model checkpoints)
# Safe to delete after the pipeline finishes.
WORK_DIR        = "/STORAGE/Arka/DC_Gaussian_Workdir"

# ══════════════════════════════════════════════════════════════════════════════
#  RUNTIME OPTIONS
# ══════════════════════════════════════════════════════════════════════════════

COLMAP_BIN    = "colmap"   # or full path: "/usr/local/bin/colmap"
USE_GPU       = True       # False → COLMAP runs on CPU (slow but works without GPU)
SKIP_EXISTING = True       # True → skip sequences that already have output files

# ══════════════════════════════════════════════════════════════════════════════
#  DC-GAUSSIAN HYPER-PARAMETERS  (from the paper — change only if needed)
# ══════════════════════════════════════════════════════════════════════════════

WIN_TYPE          = "iom"       # obstruction model type
POSITION_LR_INIT  = 0.00008     # initial position learning rate
POSITION_LR_FINAL = 0.000008    # final position learning rate
SCALING_LR        = 0.005       # scaling learning rate
PERCENT_DENSE     = 0.0005      # densification threshold
DENSIFY_UNTIL     = 20000       # stop densifying after this iteration
# Training runs for 30 000 iterations total.

# ══════════════════════════════════════════════════════════════════════════════
#  INTERNAL CONSTANTS  (do not edit)
# ══════════════════════════════════════════════════════════════════════════════

IMG_EXTS = {".jpg", ".jpeg", ".png"}   # lower-case extensions to scan

# Quick sanity print
print(f"SEQ_ROOT    : {SEQ_ROOT}")
print(f"OUTPUT_ROOT : {OUTPUT_ROOT}")
print(f"DC_GAUSSIAN : {DC_GAUSSIAN_DIR}")
print(f"WORK_DIR    : {WORK_DIR}")
print(f"COLMAP_BIN  : {COLMAP_BIN}")
print(f"USE_GPU     : {USE_GPU}  |  SKIP_EXISTING: {SKIP_EXISTING}")


## Cell 3 — Helper Utilities

Internal functions used by every pipeline step.  Run once; do not modify.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
#  run_cmd  — streams a subprocess to stdout + log file
# ─────────────────────────────────────────────────────────────────────────────
def run_cmd(cmd, cwd=None, log_file=None):
    """
    Execute `cmd` (list of strings), streaming output to stdout and
    optionally appending to `log_file`.  Raises CalledProcessError on failure.
    """
    cmd_str = [str(c) for c in cmd]
    log.info("$ %s", " ".join(cmd_str))
    fh = open(log_file, "a") if log_file else None
    try:
        proc = subprocess.Popen(
            cmd_str,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            cwd=str(cwd) if cwd else None,
            text=True,
            bufsize=1,
        )
        for line in proc.stdout:
            sys.stdout.write(line)
            sys.stdout.flush()
            if fh:
                fh.write(line)
        proc.wait()
    finally:
        if fh:
            fh.close()
    if proc.returncode != 0:
        raise subprocess.CalledProcessError(proc.returncode, cmd_str)


# ─────────────────────────────────────────────────────────────────────────────
#  get_images_sorted  — lexicographic sort (matches dashcam sequential naming)
# ─────────────────────────────────────────────────────────────────────────────
def get_images_sorted(folder: Path) -> list:
    """
    Return all image files (jpg/jpeg/png) in `folder`, sorted by filename.
    Non-recursive — only the direct children of the folder are considered.
    """
    return sorted(
        [p for p in Path(folder).iterdir()
         if p.is_file() and p.suffix.lower() in IMG_EXTS],
        key=lambda p: p.name,
    )


# ─────────────────────────────────────────────────────────────────────────────
#  parse_colmap_images_txt  — recover COLMAP camera order
# ─────────────────────────────────────────────────────────────────────────────
def parse_colmap_images_txt(images_txt: Path) -> list:
    """
    Parse COLMAP's sparse/0/images.txt and return image basenames sorted by
    ascending COLMAP image-ID.

    This order is the ground truth for which render index (00000.png,
    00001.png, …) corresponds to which source image.

    COLMAP format — alternating pairs of lines (comments start with #):
        IMAGE_ID  QW QX QY QZ  TX TY TZ  CAMERA_ID  NAME
        POINTS2D[] as  X Y POINT3D_ID
    """
    if not images_txt.exists():
        return []
    with open(images_txt) as f:
        raw = [l for l in f if not l.startswith("#") and l.strip()]
    id_to_name: dict = {}
    for i in range(0, len(raw), 2):         # skip every 2nd line (2-D points)
        parts = raw[i].strip().split()
        if len(parts) >= 10:
            id_to_name[int(parts[0])] = parts[9]
    return [id_to_name[k] for k in sorted(id_to_name)]


# ─────────────────────────────────────────────────────────────────────────────
#  build_render_to_orig_map  — render index → original Path
# ─────────────────────────────────────────────────────────────────────────────
def build_render_to_orig_map(orig_images: list, colmap_order: list) -> dict:
    """
    Build  {render_index: original_image_Path}  using the COLMAP image order.

    DC-Gaussian saves renders as  00000.png, 00001.png, …  in the same order
    as the cameras appear in COLMAP's images.txt.  We copied the originals to
    workdir/input/ with their original filenames, so COLMAP names == original
    basenames, and the mapping is exact.

    Positional fallback: if a COLMAP name cannot be matched to any original
    file (e.g. COLMAP undistortion renamed it), we fall back to the sorted
    alphabetical position — logged as a WARNING.
    """
    orig_by_name = {p.name: p for p in orig_images}
    orig_by_stem = {p.stem: p for p in orig_images}

    name_map: dict = {}
    for idx, colmap_name in enumerate(colmap_order):
        bn = Path(colmap_name).name
        st = Path(colmap_name).stem
        if bn in orig_by_name:
            name_map[idx] = orig_by_name[bn]
        elif st in orig_by_stem:
            name_map[idx] = orig_by_stem[st]

    # Positional fallback for any un-mapped indices
    assigned   = set(name_map.values())
    unassigned = [p for p in orig_images if p not in assigned]
    for idx in range(len(colmap_order)):
        if idx not in name_map and unassigned:
            name_map[idx] = unassigned.pop(0)
            log.warning("  Positional fallback for render index %d → %s",
                        idx, name_map[idx].name)
    return name_map


# ─────────────────────────────────────────────────────────────────────────────
#  find_render_dir  — locate DC-Gaussian's render output
# ─────────────────────────────────────────────────────────────────────────────
def find_render_dir(seq_model: Path):
    """
    Find the renders/ directory under seq_model.

    Priority: train/ours_{max_iter}/renders  (all frames, no --eval)
    Fallback:  test/ours_{max_iter}/renders  (only test frames, with --eval)

    Returns a Path or None.
    """
    for split in ("train", "test"):
        candidates = sorted(
            (seq_model / split).glob("ours_*/renders"),
            key=lambda p: int(p.parent.name.split("_")[-1]),
        )
        if candidates:
            return candidates[-1]   # highest iteration directory
    return None


print("Helper utilities defined ✓")


## Cell 4 — Pre-flight Check

Scans `SEQ_ROOT` and reports image counts, resolutions, and potential issues
**before** the pipeline starts.  Fix any warnings, then proceed to Cell 5.

In [ ]:
def preflight_check() -> list:
    """
    Scan SEQ_ROOT, print a per-sequence summary table, and return the list of
    sequence directories that are safe to process (≥ 10 images each).
    """
    log.info("=" * 68)
    log.info("  DC-Gaussian — Pre-flight Sequence Validation")
    log.info("  SEQ_ROOT : %s", SEQ_ROOT)
    log.info("=" * 68)

    seq_root = Path(SEQ_ROOT)
    if not seq_root.exists():
        raise FileNotFoundError(f"SEQ_ROOT not found:\n  {seq_root}")

    seq_dirs = sorted(p for p in seq_root.iterdir() if p.is_dir())
    if not seq_dirs:
        raise RuntimeError(f"No sub-directories found under:\n  {seq_root}")

    log.info("Found %d sequence(s)\n", len(seq_dirs))

    valid = []
    print(f"  {'St':2}  {'Sequence':<35}  {'#Imgs':>6}  {'Resolution':>12}  {'Notes'}")
    print("  " + "─" * 72)

    for seq in seq_dirs:
        imgs  = get_images_sorted(seq)
        n     = len(imgs)

        res_str = "n/a"
        try:
            from PIL import Image as PILImage
            with PILImage.open(imgs[0]) as im:
                res_str = f"{im.width}×{im.height}"
        except Exception:
            pass

        notes = []
        if n == 0:
            notes.append("NO IMAGES — skipped")
        elif n < 20:
            notes.append(f"only {n} imgs  (COLMAP needs ≥ 20 overlapping frames)")
        elif n > 3000:
            notes.append(f"{n} imgs — training will be slow; consider sub-sampling")

        status = "✓" if not notes else "⚠"
        note_str = ";  ".join(notes) if notes else "OK"
        print(f"  {status:2}  {seq.name:<35}  {n:>6}  {res_str:>12}  {note_str}")

        if n >= 10:
            valid.append(seq)

    print()
    log.info("Valid sequences (will be processed) : %d / %d", len(valid), len(seq_dirs))
    return valid

# ── Run it ────────────────────────────────────────────────────────────────────
valid_sequences = preflight_check()


## Cell 5 — Pipeline Step Functions

Five functions, one per stage.  They are called automatically by the main loop
in Cell 6 — no need to call them manually.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
#  STEP 1  — Copy source images → workdir/input/  (COLMAP expects them there)
# ─────────────────────────────────────────────────────────────────────────────
def step1_copy_images(seq_dir: Path, seq_work: Path, orig_images: list) -> int:
    inp_dir = seq_work / "input"
    inp_dir.mkdir(parents=True, exist_ok=True)
    copied = 0
    for img in orig_images:
        dst = inp_dir / img.name
        if not dst.exists():
            shutil.copy2(img, dst)
            copied += 1
    log.info("  [STEP 1] %d images ready in %s", copied if copied else len(orig_images), inp_dir)
    return copied


# ─────────────────────────────────────────────────────────────────────────────
#  STEP 2  — COLMAP: feature extraction → matching → SfM → undistortion
# ─────────────────────────────────────────────────────────────────────────────
def step2_colmap(seq_work: Path, log_file: Path):
    log.info("  [STEP 2] COLMAP (feature extraction → matching → SfM → undistortion) …")
    cmd = [
        sys.executable,
        str(Path(DC_GAUSSIAN_DIR) / "convert.py"),
        "--source_path",      str(seq_work),
        "--colmap_executable", COLMAP_BIN,
    ]
    if not USE_GPU:
        cmd.append("--no_gpu")
    run_cmd(cmd, cwd=DC_GAUSSIAN_DIR, log_file=log_file)
    log.info("  [STEP 2] COLMAP done ✓")


# ─────────────────────────────────────────────────────────────────────────────
#  STEP 3  — Train DC-Gaussian
#
#  NOTE: --eval is intentionally omitted.
#    • Without --eval → ALL images form the training set.
#    • This means every frame will be rendered in STEP 4.
#    • If you want a held-out test split for quantitative evaluation,
#      add "--eval" here and change STEP 4 to use "--skip_train" instead.
# ─────────────────────────────────────────────────────────────────────────────
def step3_train(seq_work: Path, seq_model: Path, log_file: Path):
    log.info("  [STEP 3] Training DC-Gaussian (~30 000 iterations) …")
    seq_model.mkdir(parents=True, exist_ok=True)
    cmd = [
        sys.executable,
        str(Path(DC_GAUSSIAN_DIR) / "train.py"),
        "-s", str(seq_work),
        "-m", str(seq_model),
        # No --eval → all images in train set → all get rendered
        "-r", "1",                               # full resolution
        "--win_type",            WIN_TYPE,
        "--position_lr_init",    str(POSITION_LR_INIT),
        "--position_lr_final",   str(POSITION_LR_FINAL),
        "--scaling_lr",          str(SCALING_LR),
        "--percent_dense",       str(PERCENT_DENSE),
        "--densify_until_iter",  str(DENSIFY_UNTIL),
    ]
    run_cmd(cmd, cwd=DC_GAUSSIAN_DIR, log_file=log_file)
    log.info("  [STEP 3] Training done ✓")


# ─────────────────────────────────────────────────────────────────────────────
#  STEP 4  — Render reflection-free images
#
#  --skip_test: because without --eval there is no test split to render.
#  Outputs land in:  seq_model/train/ours_{iter}/renders/00000.png …
# ─────────────────────────────────────────────────────────────────────────────
def step4_render(seq_work: Path, seq_model: Path, log_file: Path):
    log.info("  [STEP 4] Rendering reflection-free images (all train frames) …")
    cmd = [
        sys.executable,
        str(Path(DC_GAUSSIAN_DIR) / "render.py"),
        "-s", str(seq_work),
        "-m", str(seq_model),
        "--skip_test",   # no test split (train-only run)
    ]
    run_cmd(cmd, cwd=DC_GAUSSIAN_DIR, log_file=log_file)
    log.info("  [STEP 4] Rendering done ✓")


# ─────────────────────────────────────────────────────────────────────────────
#  STEP 5  — Collect renders, rename to ORIGINAL filenames, save to OUTPUT_ROOT
#
#  Mapping logic:
#    DC-Gaussian numbers renders as 00000.png, 00001.png, … in the same order
#    as COLMAP registered the cameras (images.txt, ascending image-ID).
#    We parse images.txt to get the exact original→render correspondence.
#    Fallback: sorted-alphabetical positional mapping (logged as WARNING).
#
#  Output filename: <original_stem>.png
#    The stem is preserved (e.g. IMG_0042.jpg → IMG_0042.png).
#    Extension is always .png because DC-Gaussian renders as PNG.
# ─────────────────────────────────────────────────────────────────────────────
def step5_collect_with_original_names(
    seq_model: Path,
    seq_clean: Path,
    orig_images: list,
    seq_work: Path,
) -> int:
    log.info("  [STEP 5] Collecting renders — mapping to original filenames …")

    # Locate rendered output directory
    render_dir = find_render_dir(seq_model)
    if render_dir is None:
        raise FileNotFoundError(
            f"No render output found under {seq_model}.  "
            "Ensure render.py completed without errors."
        )
    log.info("         Render source : %s", render_dir)

    # Gather rendered frames (sorted numerically: 00000 < 00001 < …)
    rendered = sorted(
        list(render_dir.glob("*.png")) + list(render_dir.glob("*.jpg")),
        key=lambda p: p.stem,
    )
    if not rendered:
        raise FileNotFoundError(f"Render directory exists but is empty: {render_dir}")
    log.info("         Rendered frames : %d", len(rendered))

    # Build render-index → original-Path map
    images_txt   = seq_work / "sparse" / "0" / "images.txt"
    colmap_order = parse_colmap_images_txt(images_txt)

    if colmap_order:
        log.info("         COLMAP images.txt loaded (%d entries)", len(colmap_order))
        name_map = build_render_to_orig_map(orig_images, colmap_order)
    else:
        log.warning("         images.txt missing — using positional (alphabetical) mapping")
        name_map = {i: orig_images[i]
                    for i in range(min(len(rendered), len(orig_images)))}

    seq_clean.mkdir(parents=True, exist_ok=True)

    saved = 0
    for idx, rendered_img in enumerate(rendered):
        orig = name_map.get(idx)
        if orig is None:
            log.warning("         No mapping for render index %d — skipped", idx)
            continue
        # Preserve original stem; render is always PNG
        out_path = seq_clean / (orig.stem + ".png")
        shutil.copy2(rendered_img, out_path)
        saved += 1

    log.info("  [STEP 5] %d reflection-free images saved → %s", saved, seq_clean)
    return saved


print("Pipeline step functions defined ✓")


## Cell 6 — Main Pipeline Loop

Processes every sequence end-to-end.
- Uses `valid_sequences` discovered in Cell 4.
- Resumable: if `SKIP_EXISTING = True`, completed sequences are skipped.
- All output and logs per sequence are preserved on failure; other sequences continue.

> ⏱ **Expected time per sequence**: 30–90 minutes depending on frame count and GPU.

In [ ]:
def process_sequence(seq_dir: Path, stats: dict):
    """
    Run all 5 pipeline steps for one sequence.
    Updates `stats` dict in-place with passed / failed / skipped counts.
    """
    seq_name  = seq_dir.name
    seq_work  = Path(WORK_DIR)    / "workdir" / seq_name
    seq_model = Path(WORK_DIR)    / "models"  / seq_name
    seq_clean = Path(OUTPUT_ROOT) / seq_name
    log_file  = Path(WORK_DIR)    / "logs"    / f"{seq_name}.log"
    log_file.parent.mkdir(parents=True, exist_ok=True)

    log.info("")
    log.info("─" * 60)
    log.info("  Sequence : %s", seq_name)
    log.info("─" * 60)

    # ── Skip check ────────────────────────────────────────────────────────────
    if SKIP_EXISTING and seq_clean.exists() and any(seq_clean.iterdir()):
        existing = len(list(seq_clean.glob("*.png")))
        log.info("  [SKIP] %d renders already exist → %s", existing, seq_clean)
        stats["skipped"] += 1
        return

    # ── Input images ──────────────────────────────────────────────────────────
    orig_images = get_images_sorted(seq_dir)
    log.info("  Input images (sorted by name) : %d", len(orig_images))
    if len(orig_images) < 10:
        log.warning("  Too few images (%d). Skipping.", len(orig_images))
        stats["failed"].append(seq_name)
        return

    # Initialise log file
    with open(log_file, "w") as f:
        f.write(f"=== {seq_name}  {datetime.now().isoformat()} ===\n")
        f.write(f"Input images : {len(orig_images)}\n\n")

    try:
        t_start = datetime.now()

        step1_copy_images(seq_dir, seq_work, orig_images)
        step2_colmap(seq_work, log_file)
        step3_train(seq_work, seq_model, log_file)
        step4_render(seq_work, seq_model, log_file)
        n_saved = step5_collect_with_original_names(
            seq_model, seq_clean, orig_images, seq_work
        )

        elapsed_min = (datetime.now() - t_start).seconds / 60
        log.info("  ✓  DONE — %d reflection-free images saved  (%.1f min)",
                 n_saved, elapsed_min)
        stats["passed"] += 1

    except Exception as exc:
        log.error("  ✗  FAILED : %s", exc)
        with open(log_file, "a") as f:
            f.write(f"\n[FAILED]  {exc}\n")
        stats["failed"].append(seq_name)


# ══════════════════════════════════════════════════════════════════════════════
#  RUN THE FULL PIPELINE
# ══════════════════════════════════════════════════════════════════════════════
def run_pipeline(seq_dirs=None):
    """
    Iterate over `seq_dirs` and call `process_sequence` for each.
    Returns a stats dict: {passed, failed, skipped}.
    """
    if seq_dirs is None:
        seq_dirs = sorted(p for p in Path(SEQ_ROOT).iterdir() if p.is_dir())

    for d in [OUTPUT_ROOT, WORK_DIR]:
        Path(d).mkdir(parents=True, exist_ok=True)

    log.info("=" * 68)
    log.info("  DC-Gaussian Batch Reflection-Removal Pipeline")
    log.info("  Input        : %s", SEQ_ROOT)
    log.info("  Output       : %s", OUTPUT_ROOT)
    log.info("  Sequences    : %d", len(seq_dirs))
    log.info("  SKIP_EXISTING: %s", SKIP_EXISTING)
    log.info("=" * 68)

    stats = {"passed": 0, "failed": [], "skipped": 0}
    for seq in seq_dirs:
        process_sequence(seq, stats)

    log.info("")
    log.info("=" * 68)
    log.info("  PIPELINE COMPLETE")
    log.info("  Passed  : %d", stats["passed"])
    log.info("  Failed  : %d  %s",
             len(stats["failed"]), stats["failed"] if stats["failed"] else "")
    log.info("  Skipped : %d", stats["skipped"])
    log.info("  Output  : %s", OUTPUT_ROOT)
    log.info("=" * 68)
    return stats

# ── Launch ────────────────────────────────────────────────────────────────────
pipeline_stats = run_pipeline(seq_dirs=valid_sequences)


## Cell 7 — Post-run Summary

Run this cell after the pipeline finishes (or at any point) to see a
per-sequence status table and generate a thumbnail grid (`thumbnail_grid.jpg`).

In [ ]:
def post_summary():
    """
    Print per-sequence render counts, flag failures, and optionally create
    a thumbnail grid of mid-sequence reflection-free frames.
    """
    log.info("=" * 68)
    log.info("  DC-Gaussian — Post-pipeline Output Summary")
    log.info("=" * 68)

    clean_base = Path(OUTPUT_ROOT)
    log_base   = Path(WORK_DIR) / "logs"

    if not Path(SEQ_ROOT).exists():
        log.error("SEQ_ROOT not found: %s", SEQ_ROOT)
        return

    seq_dirs = sorted(p for p in Path(SEQ_ROOT).iterdir() if p.is_dir())
    total_renders = 0
    failed_list   = []
    pending_list  = []

    print(f"\n  {'Sequence':<38}  {'Renders':>7}  Status")
    print("  " + "─" * 62)

    for seq in seq_dirs:
        name     = seq.name
        out_dir  = clean_base / name
        log_file = log_base / f"{name}.log"

        if out_dir.exists():
            renders = list(out_dir.glob("*.png")) + list(out_dir.glob("*.jpg"))
            n = len(renders)
            total_renders += n
            status = "✓ done" if n > 0 else "⚠  dir exists but empty"
            print(f"  {name:<38}  {n:>7}  {status}")
        else:
            if log_file.exists():
                txt = log_file.read_text(errors="ignore")
                if "[FAILED]" in txt or "CalledProcessError" in txt:
                    status = "✗  FAILED — check log"
                    failed_list.append(name)
                else:
                    status = "─  not yet run"
                    pending_list.append(name)
            else:
                status = "─  not yet run"
                pending_list.append(name)
            print(f"  {name:<38}  {'—':>7}  {status}")

    print()
    log.info("Total reflection-free images : %d", total_renders)
    log.info("Failed sequences             : %d", len(failed_list))
    log.info("Pending sequences            : %d", len(pending_list))
    if failed_list:
        log.warning("Failed: %s", failed_list)
        log.info("Logs: %s", log_base)

    # ── Thumbnail grid ────────────────────────────────────────────────────────
    _generate_thumbnail_grid(clean_base)


def _generate_thumbnail_grid(clean_base: Path):
    """
    Generate a JPEG grid of mid-sequence reflection-free frames.
    Requires Pillow (pip install pillow).
    """
    try:
        from PIL import Image, ImageDraw, ImageFont
    except ImportError:
        log.info("Pillow not installed — skipping thumbnail grid.")
        return

    THUMB   = (320, 180)
    COLS    = 4
    PAD     = 8
    LABEL_H = 22

    seq_dirs = sorted(p for p in clean_base.iterdir() if p.is_dir())
    thumbs   = []

    for seq in seq_dirs:
        imgs = sorted(seq.glob("*.png")) + sorted(seq.glob("*.jpg"))
        if not imgs:
            continue
        mid = imgs[len(imgs) // 2]
        try:
            im = Image.open(mid).convert("RGB")
            im.thumbnail(THUMB)
            thumbs.append((seq.name, im))
        except Exception:
            continue

    if not thumbs:
        return

    rows = (len(thumbs) + COLS - 1) // COLS
    W    = COLS * (THUMB[0] + PAD) + PAD
    H    = rows * (THUMB[1] + LABEL_H + PAD) + PAD
    grid = Image.new("RGB", (W, H), (30, 30, 30))
    draw = ImageDraw.Draw(grid)
    try:
        font = ImageFont.truetype(
            "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", 13)
    except Exception:
        font = ImageFont.load_default()

    for idx, (name, thumb) in enumerate(thumbs):
        r, c = divmod(idx, COLS)
        x = PAD + c * (THUMB[0] + PAD)
        y = PAD + r * (THUMB[1] + LABEL_H + PAD)
        grid.paste(thumb, (x, y))
        draw.text((x + 4, y + THUMB[1] + 3), name[:40],
                  fill=(220, 220, 220), font=font)

    out = Path(OUTPUT_ROOT) / "thumbnail_grid.jpg"
    grid.save(out, quality=88)
    log.info("Thumbnail grid saved → %s", out)


# ── Run ───────────────────────────────────────────────────────────────────────
post_summary()
